In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts").exists(), f"Racine projet introuvable depuis {Path.cwd()}"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("memory_growth:", e)

print("TF:", tf.__version__)


ROOT = /home/ui/PROJ9


2026-02-24 19:41:46.491025: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF: 2.20.0


In [2]:
import time, json, inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from scripts.config import CITYSCAPES_DIR, EXP_DIR, resolve_split_csv, ensure_dirs
from scripts.datagen import CityscapesSequence
from scripts.augmentations import make_train_aug
from scripts.training import compile_model, reset_between_runs
from scripts.losses_metrics import MeanIoUArgmax, dice_loss_sparse
from scripts.preprocessing import colorize_groups, overlay, N_CLASSES

ensure_dirs()
print("CITYSCAPES_DIR:", CITYSCAPES_DIR)
print("EXP_DIR:", EXP_DIR)

csv_path = resolve_split_csv()
df_idx = pd.read_csv(csv_path)

if "image_path" not in df_idx.columns:
    df_idx["image_path"] = df_idx["image_rel"].apply(lambda p: str(Path(CITYSCAPES_DIR) / p))
if "mask_path" not in df_idx.columns:
    df_idx["mask_path"] = df_idx["mask_rel"].apply(lambda p: str(Path(CITYSCAPES_DIR) / p))

train_df = df_idx[df_idx["split_final"] == "train"].copy()
val_df   = df_idx[df_idx["split_final"] == "val"].copy()
test_df  = df_idx[df_idx["split_final"] == "test"].copy()

print(df_idx["split_final"].value_counts().to_string())
print("train/val/test:", len(train_df), len(val_df), len(test_df))

import scripts.models as models
print("models loaded.")


CITYSCAPES_DIR: /home/ui/PROJ9/data/raw/cityscapes
EXP_DIR: /home/ui/PROJ9/out/experiments
split_final
train    2380
test      595
val       500
train/val/test: 2380 500 595
models loaded.


I0000 00:00:1771958513.724561 3332170 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9700 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080 Ti, pci bus id: 0000:07:00.0, compute capability: 6.1


In [3]:
def _write_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def save_curves(history: dict, run_dir: Path):
    run_dir.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 4))
    if "loss" in history: plt.plot(history["loss"], label="loss")
    if "val_loss" in history: plt.plot(history["val_loss"], label="val_loss")
    plt.legend(); plt.title("Loss"); plt.xlabel("epoch"); plt.ylabel("loss")
    plt.tight_layout()
    plt.savefig(run_dir / "loss.png", dpi=160)
    plt.close()

    plt.figure(figsize=(10, 4))
    if "mIoU" in history: plt.plot(history["mIoU"], label="mIoU")
    if "val_mIoU" in history: plt.plot(history["val_mIoU"], label="val_mIoU")
    plt.legend(); plt.title("mIoU"); plt.xlabel("epoch"); plt.ylabel("IoU")
    plt.tight_layout()
    plt.savefig(run_dir / "miou.png", dpi=160)
    plt.close()

def save_pred_grid(model, df_split: pd.DataFrame, run_dir: Path, size_hw=(512,512), n=6, seed=42, alpha=0.45):
    H, W = size_hw
    sample = df_split.sample(n=min(n, len(df_split)), random_state=seed).reset_index(drop=True)

    gap = 6
    cell_w, cell_h = W, H
    grid = Image.new("RGB", (4*cell_w + 3*gap, len(sample)*cell_h + (len(sample)-1)*gap), (0,0,0))

    for i, row in sample.iterrows():
        img = Image.open(row["image_path"]).convert("RGB").resize((W, H), Image.BILINEAR)

        x = (np.array(img, dtype=np.float32) / 255.0)[None, ...]
        pred = model.predict(x, verbose=0)[0]
        pred_cls = np.argmax(pred, axis=-1).astype(np.uint8)

        one = pd.DataFrame([row])
        seq = CityscapesSequence(one, base_dir=CITYSCAPES_DIR, batch_size=1, size_hw=size_hw,
                                 augment=None, shuffle=False, seed=seed, aug_repeats=1)
        X1, y1 = seq[0]
        gt = Image.fromarray(y1[0, :, :, 0].astype(np.uint8), mode="L")

        gt_rgb = colorize_groups(gt)
        pr_rgb = colorize_groups(Image.fromarray(pred_cls, mode="L"))
        ov = overlay(img, pr_rgb, alpha=alpha)

        y0 = i*(cell_h + gap)
        for j, pic in enumerate([img, gt_rgb, pr_rgb, ov]):
            x0 = j*(cell_w + gap)
            grid.paste(pic, (x0, y0))

    out_path = run_dir / "pred_grid.png"
    grid.save(out_path)
    return out_path

def _has_param(fn, name: str) -> bool:
    try:
        return name in inspect.signature(fn).parameters
    except Exception:
        return False


In [4]:
import tensorflow as tf
from tensorflow.keras import callbacks

def build_model(build_fn, input_shape, n_classes, trainable=None, encoder_weights="imagenet"):
    kwargs = {}
    if _has_param(build_fn, "input_shape"):
        kwargs["input_shape"] = input_shape
    if _has_param(build_fn, "n_classes"):
        kwargs["n_classes"] = n_classes
    if trainable is not None and _has_param(build_fn, "trainable"):
        kwargs["trainable"] = bool(trainable)
    if _has_param(build_fn, "encoder_weights"):
        kwargs["encoder_weights"] = encoder_weights
    return build_fn(**kwargs)

def run_one(
    tag: str,
    build_fn,
    size_hw=(512,512),
    batch=4,
    epochs=10,
    aug=True,
    loss_name="ce_dice",
    patience=3,
    seed=42,
    trainable=None,
):
    reset_between_runs(seed)

    train_aug = make_train_aug() if aug else None

    train_seq = CityscapesSequence(train_df, base_dir=CITYSCAPES_DIR, batch_size=batch, size_hw=size_hw,
                                   augment=train_aug, shuffle=True, seed=seed, aug_repeats=1)
    val_seq   = CityscapesSequence(val_df, base_dir=CITYSCAPES_DIR, batch_size=batch, size_hw=size_hw,
                                   augment=None, shuffle=False, seed=seed, aug_repeats=1)
    test_seq  = CityscapesSequence(test_df, base_dir=CITYSCAPES_DIR, batch_size=batch, size_hw=size_hw,
                                   augment=None, shuffle=False, seed=seed, aug_repeats=1)

    model = build_model(build_fn, input_shape=(size_hw[0], size_hw[1], 3), n_classes=N_CLASSES, trainable=trainable)
    model = compile_model(model, loss_name=loss_name, lr=3e-4)

    if trainable is None:
        run_name = f"{tag}_{size_hw[0]}x{size_hw[1]}_b{batch}_aug{int(aug)}_{loss_name}_e{epochs}_seed{seed}"
    else:
        run_name = f"{tag}_{'FT' if trainable else 'FROZEN'}_{size_hw[0]}x{size_hw[1]}_b{batch}_aug{int(aug)}_{loss_name}_e{epochs}_seed{seed}"

    run_dir = Path(EXP_DIR) / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path = run_dir / "best.keras"

    cb = [
        callbacks.ModelCheckpoint(str(best_path), monitor="val_mIoU", mode="max", save_best_only=True),
        callbacks.EarlyStopping(monitor="val_mIoU", mode="max", patience=patience, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_mIoU", mode="max", patience=2, factor=0.5, min_lr=3e-4),
    ]

    t0 = time.time()
    hist = model.fit(train_seq, validation_data=val_seq, epochs=epochs, callbacks=cb, verbose=1)
    t_train = time.time() - t0

    val_eval  = model.evaluate(val_seq, verbose=0)
    test_eval = model.evaluate(test_seq, verbose=0)

    summary = {
        "run_name": run_name,
        "best_path": str(best_path),
        "train_time_sec": float(t_train),
        "val_loss": float(val_eval[0]),
        "val_mIoU": float(val_eval[1]),
        "test_loss": float(test_eval[0]),
        "test_mIoU": float(test_eval[1]),
        "params": {
            "tag": tag,
            "size_hw": list(size_hw),
            "batch": int(batch),
            "epochs": int(epochs),
            "aug": bool(aug),
            "aug_repeats": 1,
            "loss_name": loss_name,
            "seed": int(seed),
            "trainable": None if trainable is None else bool(trainable),
        },
    }

    _write_json(summary, run_dir / "summary.json")
    _write_json(hist.history, run_dir / "history.json")
    save_curves(hist.history, run_dir)
    save_pred_grid(model, val_df, run_dir, size_hw=size_hw, n=6, seed=seed)

    return summary


In [5]:
SIZE_HW = (512, 512)
EPOCHS = 10
BATCH = 4
AUG = True
LOSS = "ce_dice"
PATIENCE = 3
SEED = 42

wanted = [
    ("UNET_SCRATCH", "unet_scratch", None),
    ("VGG16", "unet_vgg16", False),
    ("VGG16", "unet_vgg16", True),
    ("RESNET50", "unet_resnet50", False),
    ("RESNET50", "unet_resnet50", True),
    ("CONVNEXT_TINY", "unet_convnext_tiny", False),
    ("CONVNEXT_TINY", "unet_convnext_tiny", True),
    ("SEGFORMER_MITB0", "segformer_mitb0", False),
    ("SEGFORMER_MITB0", "segformer_mitb0", True),
]

results = []
for tag, fn_name, tr in wanted:
    if not hasattr(models, fn_name):
        print("SKIP (missing):", fn_name)
        continue
    fn = getattr(models, fn_name)
    print("\nRUN:", tag, fn_name, "trainable=", tr)
    s = run_one(
        tag=tag,
        build_fn=fn,
        size_hw=SIZE_HW,
        batch=BATCH,
        epochs=EPOCHS,
        aug=AUG,
        loss_name=LOSS,
        patience=PATIENCE,
        seed=SEED,
        trainable=tr,
    )
    results.append(s)

df = pd.DataFrame([{
    "run_name": r["run_name"],
    "val_mIoU": r["val_mIoU"],
    "test_mIoU": r["test_mIoU"],
    "val_loss": r["val_loss"],
    "test_loss": r["test_loss"],
    "train_time_sec": r["train_time_sec"],
    "best_path": r["best_path"],
} for r in results]).sort_values("val_mIoU", ascending=False)

out_csv = Path(EXP_DIR) / "bench_512_e10_summary.csv"
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)
df


/home/ui/PROJ9/scripts/augmentations.py:15: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),



RUN: UNET_SCRATCH unet_scratch trainable= None


/home/ui/PROJ9/.env0/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Epoch 1/10


2026-02-24 19:42:06.156106: I external/local_xla/xla/service/service.cc:163] XLA service 0x78f98802f430 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-24 19:42:06.156145: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1080 Ti, Compute Capability 6.1
2026-02-24 19:42:06.387184: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-24 19:42:07.131019: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/loss/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-02-24 19:42:07.770552: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-02-24 19:42:11.344522: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations

...

2026-02-25 06:44:57.251548: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_1', 12 bytes spill stores, 12 bytes spill loads



595/595 ━━━━━━━━━━━━━━━━━━━━ 377s 567ms/step - loss: 0.6758 - mIoU: 0.5920 - val_loss: 0.4022 - val_mIoU: 0.7226 - learning_rate: 3.0000e-04
Epoch 2/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 220s 369ms/step - loss: 0.5659 - mIoU: 0.6458 - val_loss: 0.3802 - val_mIoU: 0.7342 - learning_rate: 3.0000e-04
Epoch 3/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 296s 497ms/step - loss: 0.5258 - mIoU: 0.6630 - val_loss: 0.3696 - val_mIoU: 0.7390 - learning_rate: 3.0000e-04
Epoch 4/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 220s 369ms/step - loss: 0.5372 - mIoU: 0.6566 - val_loss: 0.3631 - val_mIoU: 0.7429 - learning_rate: 3.0000e-04
Epoch 5/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 293s 493ms/step - loss: 0.5055 - mIoU: 0.6718 - val_loss: 0.3585 - val_mIoU: 0.7451 - learning_rate: 3.0000e-04
Epoch 6/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 254s 427ms/step - loss: 0.5083 - mIoU: 0.6715 - val_loss: 0.3557 - val_mIoU: 0.7453 - learning_rate: 3.0000e-04
Epoch 7/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 256s 430ms/step - loss: 0.5069 - mIoU: 0.6712 - val_loss: 0

2026-02-25 07:31:05.884705: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[4,32,128,128]{3,2,1,0}, u8[0]{0}) custom-call(f32[4,32,16,16]{3,2,1,0}, f32[32,32,8,8]{3,2,1,0}), window={size=8x8 stride=8x8}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-02-25 07:31:06.509142: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.62450684s
Trying algorithm eng0{} for conv (f32[4,32,128,128]{3,2,1,0}, u8[0]{0}) custom-call(f32[4,32,16,16]{3,2,1,0}, f32[32,32,8,8]{3,2,1,0}), window={size=8x8 stride=8x8}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"operation_queue_id":"0"

595/595 ━━━━━━━━━━━━━━━━━━━━ 352s 456ms/step - loss: 0.5749 - mIoU: 0.6317 - val_loss: 0.3940 - val_mIoU: 0.7212 - learning_rate: 3.0000e-04
Epoch 2/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 221s 372ms/step - loss: 0.4501 - mIoU: 0.6907 - val_loss: 0.3809 - val_mIoU: 0.7308 - learning_rate: 3.0000e-04
Epoch 3/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 231s 388ms/step - loss: 0.4158 - mIoU: 0.7090 - val_loss: 0.3691 - val_mIoU: 0.7367 - learning_rate: 3.0000e-04
Epoch 4/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 251s 421ms/step - loss: 0.4036 - mIoU: 0.7149 - val_loss: 0.3758 - val_mIoU: 0.7355 - learning_rate: 3.0000e-04
Epoch 5/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 238s 400ms/step - loss: 0.3976 - mIoU: 0.7180 - val_loss: 0.3570 - val_mIoU: 0.7442 - learning_rate: 3.0000e-04
Epoch 6/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 237s 397ms/step - loss: 0.3751 - mIoU: 0.7301 - val_loss: 0.3476 - val_mIoU: 0.7519 - learning_rate: 3.0000e-04
Epoch 7/10
595/595 ━━━━━━━━━━━━━━━━━━━━ 224s 376ms/step - loss: 0.3797 - mIoU: 0.7285 - val_loss: 0

,run_name,val_mIoU,test_mIoU,val_loss,test_loss,train_time_sec,best_path
6,CONVNEXT_TINY_FT_512x512_b4_aug1_ce_dice_e10_s...,0.814141,0.825904,0.263996,0.235353,7807.333606,/home/ui/PROJ9/out/experiments/CONVNEXT_TINY_F...
5,CONVNEXT_TINY_FROZEN_512x512_b4_aug1_ce_dice_e...,0.782934,0.793668,0.308337,0.280866,9680.691940,/home/ui/PROJ9/out/experiments/CONVNEXT_TINY_F...
3,RESNET50_FROZEN_512x512_b4_aug1_ce_dice_e10_se...,0.770636,0.784616,0.327380,0.294545,2984.075763,/home/ui/PROJ9/out/experiments/RESNET50_FROZEN...
2,VGG16_FT_512x512_b4_aug1_ce_dice_e10_seed42,0.759461,0.778235,0.352077,0.309833,6732.097547,/home/ui/PROJ9/out/experiments/VGG16_FT_512x51...
8,SEGFORMER_MITB0_FT_512x512_b4_aug1_ce_dice_e10...,0.757331,0.773042,0.343468,0.305802,2482.434545,/home/ui/PROJ9/out/experiments/SEGFORMER_MITB0...
7,SEGFORMER_MITB0_FROZEN_512x512_b4_aug1_ce_dice...,0.752727,0.767666,0.346712,0.308438,2654.814470,/home/ui/PROJ9/out/experiments/SEGFORMER_MITB0...
4,RESNET50_FT_512x512_b4_aug1_ce_dice_e10_seed42,0.712800,0.731461,0.431505,0.388337,2267.004815,/home/ui/PROJ9/out/experiments/RESNET50_FT_512...
1,VGG16_FROZEN_512x512_b4_aug1_ce_dice_e10_seed42,0.703438,0.726915,0.445701,0.398493,6796.812620,/home/ui/PROJ9/out/experiments/VGG16_FROZEN_51...
0,UNET_SCRATCH_512x512_b4_aug1_ce_dice_e10_seed42,0.689710,0.707698,0.464114,0.418740,2547.849639,/home/ui/PROJ9/out/experiments/UNET_SCRATCH_51...
